# 02 - Train ResNet Ensemble

Step 02 of the TamIA pipeline in Colab format. Trains the 5 ResNet-18 verifier models on the regenerated hard-negative dataset.

Run this notebook with a GPU runtime. The code executes the same root script used by the TamIA Slurm job.

In [ ]:
#@title Colab setup
from pathlib import Path
import os
import subprocess
import sys

# Edit these if needed.
REPO_URL = "https://github.com/moradBMH/INF8225_Projet.git"
GIT_REF = "temp"
PROJECT_DIR = Path("/content/INF8225_Projet")

# If your Drive folder is not auto-detected by colab/setup.py, set it explicitly,
# for example: "/content/drive/MyDrive/Projet_Medsam".
DRIVE_FOLDER = None

# Keep INSTALL_DEPS=True on the first notebook in a fresh Colab runtime.
# You can set it to False for later notebooks in the same runtime.
INSTALL_DEPS = True
REINSTALL_DEPS = False

if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", "--branch", GIT_REF, REPO_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "fetch", "origin", GIT_REF], check=False)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", GIT_REF], check=False)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull"], check=False)

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from colab.setup import setup
setup(drive_folder=DRIVE_FOLDER, install=INSTALL_DEPS, reinstall=REINSTALL_DEPS)
print("cwd:", Path.cwd())


In [ ]:
#@title Verify shared assets
from pathlib import Path

required = [
    Path("data/MSD_pancreas/train.json"),
    Path("data/MSD_pancreas/val.json"),
    Path("data/MSD_pancreas/test.json"),
    Path("work_dirs/tumor_config_v3/best_coco_bbox_mAP_epoch_25.pth"),
    Path("work_dirs/tumor_config_v3/tumor_config_v3.py"),
]
missing = [p for p in required if not p.exists()]
for p in required:
    print(("OK     " if p.exists() else "MISSING"), p)
assert not missing, f"Missing required files: {missing}"

required += [
    Path("data/classifier_dataset_hard/train/0"),
    Path("data/classifier_dataset_hard/train/1"),
    Path("data/classifier_dataset_hard/val/0"),
    Path("data/classifier_dataset_hard/val/1"),
]
missing = [p for p in required if not p.exists()]
for p in required:
    print(("OK     " if p.exists() else "MISSING"), p)
assert not missing, f"Missing required files/directories: {missing}"


In [ ]:
#@title Run pipeline step
import subprocess
import sys
subprocess.run([sys.executable, "-u", "train_resnet_kfold_recall.py"], check=True)


In [ ]:
#@title Inspect ResNet artifacts
from pathlib import Path
for i in range(1, 6):
    ckpt = Path(f"resnet_fold_{i}.pth")
    thr = Path(f"threshold_run_{i}.txt")
    print(f"fold {i}: ckpt={ckpt.exists()} threshold={thr.exists()}")
assert all(Path(f"resnet_fold_{i}.pth").exists() for i in range(1, 6)), "Missing ResNet checkpoint(s)"
